In [2]:
# Imports
import pandas as pd
import numpy as np
import os


In [ ]:
# Importing DB...
BASE_DB = "mimic-iv-clinical-database-demo-2.2"
HOSP_PATH = os.path.join(BASE_DB, "hosp")
ICU_PATH = os.path.join(BASE_DB, "icu")

# Loading Tables...
patients = pd.read_csv(f'{HOSP_PATH}/patients.csv.gz')
admissions = pd.read_csv(f'{HOSP_PATH}/admissions.csv.gz')
lab_events = pd.read_csv(f'{HOSP_PATH}/labevents.csv.gz')

chart_events = pd.read_csv(f'{ICU_PATH}/chartevents.csv.gz')
icu_stays = pd.read_csv(f'{ICU_PATH}/icustays.csv.gz')
d_items = pd.read_csv(f'{ICU_PATH}/d_items.csv.gz') 



array([ 0.,  1., nan])

# Step 0 : Data Exploring

In [ ]:
######## Charevents table: understanding clinical variables
# each itemid represents the SAME clinical measurements in the ICU system (like: heart rate, BP,Respiratory,...)
# For example: the heart rate measurements could be mapped to multiple item_ids (for the same patient & icu stay)
df_vitals = chart_events[
    (chart_events["valueuom"].str.lower() == "bpm") &
    (chart_events["stay_id"] == 32604416)
]
df_unique_itemids = df_vitals.drop_duplicates(subset=["itemid"])
print(df_unique_itemids[["subject_id","stay_id","itemid", "valuenum"]].head())


    subject_id   stay_id  itemid  valuenum
5     10005817  32604416  224751      52.0
11    10005817  32604416  220047      55.0
36    10005817  32604416  220046     120.0
69    10005817  32604416  220045      80.0


# Step 1: Data Extraction | Feature Definition

In [ ]:
#convert to datetime
icu_stays['intime'] = pd.to_datetime(icu_stays['intime'])
icu_stays['outtime'] = pd.to_datetime(icu_stays['outtime'])

# Select first ICU stay per patient
icu_stays = icu_stays.sort_values(by="intime")
icu_stay_first = icu_stays.groupby("subject_id").first()

# add the patients' demographics based on the chosen icu stay
df = icu_stay_first.merge(patients, on="subject_id")

# add relevant hospital admission
df = df.merge(admissions, on=["subject_id", "hadm_id"])

####### Define chartevents for 24H window (var:24H_endtime)
# Calculate the 24H_endtime based on each icu_stay intime
# Extract the chartevents done in those 24H
df["24H_endtime"] = df["intime"] + pd.Timedelta(hours=24)
chart_events["charttime"] = pd.to_datetime(chart_events["charttime"])
chart_events = chart_events.merge(df[["stay_id", "intime", "24H_endtime"]], on="stay_id")
chart_events_24H = chart_events[
    (chart_events["charttime"] >= chart_events["intime"]) &
    (chart_events["charttime"] <= chart_events["24H_endtime"])
]





# Keep relevant columns
# df = df[[
#      "subject_id","hadm_id", 
#      "stay_id", "first_careunit",
#      "anchor_age", "gender",
#      "admission_type", "hospital_expire_flag",
#      "intime","24H_endtime", "outtime"
# ]]


#df.head()

,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning,intime_x,24H_endtime_x,intime_y,24H_endtime_y,intime,24H_endtime
0,10005817,20626031,32604416,6770.0,2132-12-16,2132-12-15 23:45:00,225054,On,NaN,NaN,0.0,2132-12-15 09:29:01,2132-12-16 09:29:01,2132-12-15 09:29:01,2132-12-16 09:29:01,2132-12-15 09:29:01,2132-12-16 09:29:01
1,10005817,20626031,32604416,6770.0,2132-12-16,2132-12-15 23:43:00,223769,100,100.0,%,0.0,2132-12-15 09:29:01,2132-12-16 09:29:01,2132-12-15 09:29:01,2132-12-16 09:29:01,2132-12-15 09:29:01,2132-12-16 09:29:01
2,10005817,20626031,32604416,6770.0,2132-12-16,2132-12-15 23:47:00,223956,Atrial demand,NaN,NaN,0.0,2132-12-15 09:29:01,2132-12-16 09:29:01,2132-12-15 09:29:01,2132-12-16 09:29:01,2132-12-15 09:29:01,2132-12-16 09:29:01
3,10005817,20626031,32604416,6770.0,2132-12-16,2132-12-15 23:47:00,224866,Yes,NaN,NaN,0.0,2132-12-15 09:29:01,2132-12-16 09:29:01,2132-12-15 09:29:01,2132-12-16 09:29:01,2132-12-15 09:29:01,2132-12-16 09:29:01
4,10005817,20626031,32604416,6770.0,2132-12-16,2132-12-15 23:45:00,227341,No,0.0,NaN,0.0,2132-12-15 09:29:01,2132-12-16 09:29:01,2132-12-15 09:29:01,2132-12-16 09:29:01,2132-12-15 09:29:01,2132-12-16 09:29:01
